# Final Project: PySpark Streaming Predictions
## Jacob A. Fericy

In [1]:
from pathlib import Path
import shutil
import pandas as pd

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, Binarizer, StringIndexer, OneHotEncoder, VectorAssembler, PCA
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

## 1. Setup and File Path Definitions

In [2]:
spark = SparkSession.builder.appName("FinalProjectFericy").getOrCreate()
spark.sparkContext.setLogLevel("WARN")

#setting paths relative to current directory
BASE_DIR = Path.cwd()
ML_FILE = BASE_DIR / "power_ml_data.csv"
STREAMING_FILE = BASE_DIR / "power_streaming_data.csv"
STREAM_INPUT_DIR = BASE_DIR / "power_stream_input"

#printing output as we go
print("Current folder:", BASE_DIR)
print("Modeling file exists:", ML_FILE.exists())
print("Streaming source file exists:", STREAMING_FILE.exists())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/30 16:34:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/30 16:34:22 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Current folder: /home/jupyter-jafericy@ncsu.edu
Modeling file exists: True
Streaming source file exists: True


## 2. Load and Inspect Data

In [3]:
#reading in ml data and checking shape/head
power_pd = pd.read_csv(ML_FILE)
print(power_pd.shape)
power_pd.head()

(47174, 10)


,Temperature,Humidity,Wind_Speed,General_Diffuse_Flows,Diffuse_Flows,Power_Zone_1,Power_Zone_2,Power_Zone_3,Month,Hour
0,6.559,73.8,0.083,0.051,0.119,34055.69620,16128.87538,20240.96386,1,0
1,6.414,74.5,0.083,0.070,0.085,29814.68354,19375.07599,20131.08434,1,0
2,6.313,74.5,0.080,0.062,0.100,29128.10127,19006.68693,19668.43373,1,0
3,6.121,75.0,0.083,0.091,0.096,28228.86076,18361.09422,18899.27711,1,0
4,5.921,75.7,0.081,0.048,0.085,27335.69620,17872.34043,18442.40964,1,0


In [4]:
#check data types
power_pd.dtypes

Temperature              float64
Humidity                 float64
Wind_Speed               float64
General_Diffuse_Flows    float64
Diffuse_Flows            float64
Power_Zone_1             float64
Power_Zone_2             float64
Power_Zone_3             float64
Month                      int64
Hour                       int64
dtype: object

## 3. Convert Pandas Data Frame to Spark

In [6]:
#converting pandas df to spark and checking sample
power_sdf = spark.createDataFrame(power_pd)
power_sdf.printSchema()
power_sdf.show(5)

root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- Power_Zone_3: double (nullable = true)
 |-- Month: long (nullable = true)
 |-- Hour: long (nullable = true)



+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|Power_Zone_3|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+------------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538| 20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599| 20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693| 19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422| 18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043| 18442.40964|    1|   0|
+-----------+--------+----------+-------

## 4. Feature Engineering and Modeling Setup

In [7]:
pca_input_cols = ["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows"]

#casting hour/month and renaming target to label
sql_transformer = SQLTransformer(statement="""
    SELECT
        CAST(Hour AS DOUBLE) AS Hour_Double,
        CAST(Month AS DOUBLE) AS Month_Double,
        Temperature,
        Humidity,
        Wind_Speed,
        General_Diffuse_Flows,
        Diffuse_Flows,
        Power_Zone_1,
        Power_Zone_2,
        Power_Zone_3 AS label
    FROM __THIS__
""")

#setting up binarization, encoding, and pca feature pipeline pieces
hour_binarizer = Binarizer(threshold = 6.5, inputCol = "Hour_Double", outputCol = "Hour_Binary")
month_indexer = StringIndexer(inputCol = "Month_Double", outputCol = "Month_Index", handleInvalid = "keep")
month_encoder = OneHotEncoder(inputCols = ["Month_Index"], outputCols = ["Month_OHE"], handleInvalid = "keep")
pca_assembler = VectorAssembler(inputCols = pca_input_cols, outputCol = "pca_input_features", handleInvalid = "keep")
pca = PCA(k = 2, inputCol = "pca_input_features", outputCol = "pca_features")

#assembling final feature vector for model
feature_assembler = VectorAssembler(
    inputCols = ["pca_features", "Hour_Binary", "Power_Zone_1", "Power_Zone_2", "Month_OHE"],
    outputCol = "features",
    handleInvalid = "keep"
)

#define the model
lr = LinearRegression(featuresCol = "features", labelCol = "label", predictionCol = "prediction")

#building full piepline from transforms to regression model
pipeline = Pipeline(stages=[
    sql_transformer,
    hour_binarizer,
    month_indexer,
    month_encoder,
    pca_assembler,
    pca,
    feature_assembler,
    lr
])

## 5. Model Tuning and Cross Validation Setup

In [8]:
#predefined
reg_grid = [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]
elastic_grid = [0, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.98, 0.99, 1]

#setting grid of reg and elastic net params for tuning
param_grid = ParamGridBuilder().addGrid(lr.regParam, reg_grid).addGrid(lr.elasticNetParam, elastic_grid).build()

#setting rmse evaluator for model performance
evaluator = RegressionEvaluator(labelCol = "label", predictionCol = "prediction", metricName = "rmse")

#setting up 5 fold cv with pipeline and param grid
cv = CrossValidator(
    estimator = pipeline,
    estimatorParamMaps = param_grid,
    evaluator = evaluator,
    numFolds = 5,
    seed = 123,
    parallelism = 2
)

cv_model = cv.fit(power_sdf)
print("cross-vali complete and fitted.")

26/04/30 16:34:36 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
26/04/30 16:34:36 WARN Instrumentation: [78a8a583] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 16:34:36 WARN Instrumentation: [656e0191] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 16:34:37 WARN Instrumentation: [78a8a583] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/30 16:34:37 WARN Instrumentation: [656e0191] Cholesky solver failed due to singular covariance matrix. Retrying with Quasi-Newton solver.
26/04/30 16:34:40 WARN Instrumentation: [326fb105] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 16:34:40 WARN Instrumentation: [13e8b681] regParam is zero, which might cause numerical instability and overfitting.
26/04/30 16:34:40 W

Cross-validation model fit complete.


## 6. Output Optimal Tuning Parameters and Errors

In [10]:
#pulling best model and final lr stage from cv results
best_pipeline_model = cv_model.bestModel
best_lr_model = best_pipeline_model.stages[-1]

#grabbing best params and lowest cv rmse from model
best_reg_param = best_lr_model._java_obj.getRegParam()
best_elastic_net_param = best_lr_model._java_obj.getElasticNetParam()
best_cv_rmse = min(cv_model.avgMetrics)

print(f"best regParam: {best_reg_param}")
print(f"best elasticNetParam: {best_elastic_net_param}")
print(f"best CV RMSE: {best_cv_rmse}")

Best regParam: 0.75
Best elasticNetParam: 0.95
Best CV RMSE: 2147.797767956872


In [11]:
#running best model on training data and computing rmse
train_predictions = best_pipeline_model.transform(power_sdf)
training_rmse = evaluator.evaluate(train_predictions)
print(f"Training set RMSE: {training_rmse}")

Training set RMSE: 2147.0995345275333


In [12]:
#creating residuals and showing sample output
#seems to be some noise on resids but will check model
residuals_df = train_predictions.withColumn("residual", col("label") - col("prediction")).select("label", "prediction", "residual")
residuals_df.show(20, truncate=False)

+-----------+------------------+------------------+
|label      |prediction        |residual          |
+-----------+------------------+------------------+
|20240.96386|20877.901634045596|-636.9377740455966|
|20131.08434|18655.143745749338|1475.9405942506637|
|19668.43373|18199.787808285448|1468.645921714553 |
|18899.27711|17585.98395125972 |1313.2931587402782|
|18442.40964|16992.80071736634 |1449.6089226336626|
|18130.12048|16513.374606417226|1616.745873582775 |
|17945.06024|16089.119526848915|1855.9407131510834|
|17459.27711|15718.695724919553|1740.5813850804461|
|17025.54217|15267.226911420412|1758.3152585795888|
|16794.21687|14934.636446554136|1859.5804234458647|
|16638.07229|14649.000567913767|1989.0717220862334|
|16395.18072|14411.608186961199|1983.5725330388013|
|16117.59036|14079.473163496064|2038.1171965039357|
|15822.6506 |13621.669351478618|2200.9812485213824|
|15672.28916|13447.311240711413|2224.977919288587 |
|15597.10843|13299.439940967048|2297.6684890329525|
|15510.36145

In [13]:
#writing placeholder producer script file to disk
prod_code = "produce_stream.py."
prod_path = BASE_DIR / "produce_stream.py"
prod_path.write_text(prod_code)
print(f"Wrote producer script to: {prod_path}")

Wrote producer script to: /home/jupyter-jafericy@ncsu.edu/produce_stream.py


In [14]:
#resetting stream input folder by deleting and recreating it. ran into an issues and think this is what you wanted anyway
if STREAM_INPUT_DIR.exists():
    shutil.rmtree(STREAM_INPUT_DIR)
STREAM_INPUT_DIR.mkdir(parents = True, exist_ok = True)
print("stream input folder:", STREAM_INPUT_DIR)

stream input folder: /home/jupyter-jafericy@ncsu.edu/power_stream_input


In [15]:
#defining schema and setting up stream to read incoming csv files
stream_schema = power_sdf.schema
stream_raw = spark.readStream.schema(stream_schema).option("header", True).csv(str(STREAM_INPUT_DIR))
print("stream created but will read new CSV files placed in:", STREAM_INPUT_DIR)

stream created but will read new CSV files placed in: /home/jupyter-jafericy@ncsu.edu/power_stream_input


## 7. Stream Transforms and Joins

In [16]:
#applying model to stream and computing residuals
prediction_stream = (
    best_pipeline_model
    .transform(stream_raw)
    .withColumn("residual", col("label") - col("prediction"))
    .select(col("label").alias("pred_label"), col("prediction"), col("residual"))
)

label_stream = stream_raw.withColumnRenamed("Power_Zone_3", "label").select(col("label"))

joined_stream = (
    prediction_stream
    .join(label_stream, prediction_stream.pred_label == label_stream.label, how="inner")
    .select(col("label"), col("prediction"), col("residual"))
)

## 8. Start Stream Outputs and End Stream Once Complete

In [17]:
#stopping old streams and starting new console output stream
for active_query in spark.streams.active:
    active_query.stop()

query = (
    joined_stream
    .writeStream
    .format("console")
    .outputMode("append")
    .option("truncate", False)
    .start()
)

print("stream start.")
print("termainal run:")
print("python produce_stream.py")

stream start.
termainal run:
python produce_stream.py


26/04/30 16:41:33 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-a27390a3-e517-4c71-b269-8cacc9035697. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/04/30 16:41:33 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [18]:
#need it to stop
query.stop()
print("stream stop.")

26/04/30 16:41:33 WARN DAGScheduler: Failed to cancel job group c6b2b3ac-a0a3-4415-a4f6-404da6d55433. Cannot find active jobs for it.


stream stop.


26/04/30 16:41:33 WARN DAGScheduler: Failed to cancel job group c6b2b3ac-a0a3-4415-a4f6-404da6d55433. Cannot find active jobs for it.
